# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# alchemy dataset
import pandas as pd
import os

data_dir = "/data/data/Alchemy-v20191129"
csv_label_path = f"{data_dir}/final_version.csv"

label = pd.read_csv(csv_label_path)

list_atom9 = os.listdir(f"{data_dir}/atom_9")
list_atom10 = os.listdir(f"{data_dir}/atom_10")
list_atom11 = os.listdir(f"{data_dir}/atom_11")
list_atom12 = os.listdir(f"{data_dir}/atom_12")

print(len(list_atom9), len(list_atom10), len(list_atom11), len(list_atom12), len(label))

np.random.shuffle(list_atom9)
np.random.shuffle(list_atom10)
np.random.shuffle(list_atom11)
np.random.shuffle(list_atom12)

num_sample = 250
list_atom9 = list_atom9[:num_sample]
list_atom10 = list_atom10[:num_sample]
list_atom11 = list_atom11[:num_sample]
list_atom12 = list_atom12[:num_sample]


35855 119661 37732 9331 202579


In [3]:
# get columns

label.columns

Index(['gdb_idx', 'atom number', 'zpve\n(Ha, zero point vibrational energy)',
       'Cv\n(cal/molK, heat capacity at 298.15 K)', 'gap\n(Ha, LUMO-HOMO)',
       'G\n(Ha, Free energy at 298.15 K)', 'HOMO\n(Ha, energy of HOMO)',
       'U\n(Ha, internal energy at 298.15 K)',
       'alpha\n(a_0^3, Isotropic polarizability)',
       'U0\n(Ha, internal energy at 0 K)', 'H\n(Ha, enthalpy at 298.15 K)',
       'LUMO\n(Ha, energy of LUMO)', 'mu\n(D, dipole moment)',
       'R2\n(a_0^2, electronic spatial extent)'],
      dtype='object')

In [4]:
def get_atom_dict(
        list_alchemy_mol,
        dir_name,
        data_dir="/data/data/Alchemy-v20191129"
):
    list_mol = []
    list_label_homo = []
    list_label_lumo = []
    list_label_gap = []
    omitted_idx = []

    for i in range(len(list_alchemy_mol)):
        gdb_idx = list_alchemy_mol[i].split(".")[0]
        sdf_path = f"{data_dir}/{dir_name}/{list_alchemy_mol[i]}"
        try:
            mol = Chem.SDMolSupplier(sdf_path)
            list_mol.append(mol)
            label_idx = label[label['gdb_idx'] == int(gdb_idx)].index
            homo_value = label['HOMO\n(Ha, energy of HOMO)'][label_idx].item()
            lumo_value = label['LUMO\n(Ha, energy of LUMO)'][label_idx].item()
            gap_value = label['gap\n(Ha, LUMO-HOMO)'][label_idx].item()
            list_label_homo.append(homo_value)
            list_label_lumo.append(lumo_value)
            list_label_gap.append(gap_value)
        except:
            omitted_idx.append(i)

    print(len(list_mol), len(omitted_idx), len(list_label_homo), len(list_label_lumo), len(list_label_gap))
    return {
        'mol': list_mol,
        'homo': list_label_homo,
        'lumo': list_label_lumo,
        'gap': list_label_gap,
    }

In [5]:
atom9_dict = get_atom_dict(
    list_alchemy_mol=list_atom9,
    dir_name="atom_9"
)
atom10_dict = get_atom_dict(
    list_alchemy_mol=list_atom10,
    dir_name="atom_10"
)
atom11_dict = get_atom_dict(
    list_alchemy_mol=list_atom11,
    dir_name="atom_11"
)
atom12_dict = get_atom_dict(
    list_alchemy_mol=list_atom12,
    dir_name="atom_12"
)

250 0 250 250 250
250 0 250 250 250
250 0 250 250 250
250 0 250 250 250


In [6]:
# concat atom dicts from 9, 10, 11, 12
list_alchemy_te_mol = atom9_dict['mol'] + atom10_dict['mol'] + atom11_dict['mol'] + atom12_dict['mol']
list_alchemy_te_mol = [i[0] for i in list_alchemy_te_mol]
list_alchemy_te_label_homo = atom9_dict['homo'] + atom10_dict['homo'] + atom11_dict['homo'] + atom12_dict['homo']
list_alchemy_te_label_lumo = atom9_dict['lumo'] + atom10_dict['lumo'] + atom11_dict['lumo'] + atom12_dict['lumo']
list_alchemy_te_label_homo_lumo_gap = atom9_dict['gap'] + atom10_dict['gap'] + atom11_dict['gap'] + atom12_dict['gap']

In [7]:
len(list_alchemy_te_mol), len(list_alchemy_te_label_homo), len(list_alchemy_te_label_lumo), len(list_alchemy_te_label_homo_lumo_gap)

(1000, 1000, 1000, 1000)

In [8]:
list_alchemy_homo_data = get_data_list(
    list_mol=list_alchemy_te_mol,
    list_label=list_alchemy_te_label_homo,
    task="alchemy_homo",
    instruction_templates=instructions_smol.qm9_homo,
)

list_alchemy_lumo_data = get_data_list(
    list_mol=list_alchemy_te_mol,
    list_label=list_alchemy_te_label_lumo,
    task="alchemy_lumo",
    instruction_templates=instructions_smol.qm9_lumo,
)

list_alchemy_homo_lumo_gap_data = get_data_list(
    list_mol=list_alchemy_te_mol,
    list_label=list_alchemy_te_label_homo_lumo_gap,
    task="alchemy_homo_lumo_gap",
    instruction_templates=instructions_smol.qm9_homo_lumo_gap,
)


100%|██████████| 1000/1000 [00:01<00:00, 725.98it/s]


In [9]:
alchemy_te_data = list_alchemy_homo_data + list_alchemy_lumo_data + list_alchemy_homo_lumo_gap_data
alchemy_te_dataset = datasets.Dataset.from_list(alchemy_te_data)
alchemy_te_dataset.save_to_disk(f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy1k_0405")

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:00<00:00, 14389.24 examples/s]


In [10]:
list_alchemy_homo_data[0]

{'task': 'alchemy_homo',
 'x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 1, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
        [5, 2, 4, 5, 0, 0, 2, 0, 1],
        [5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 0, 2, 5, 0, 0, 0, 0, 0],
        [5, 0, 2, 5, 1, 0, 0, 0, 0],
        [5, 2, 4, 5, 1, 0, 2, 0, 1],
        [6, 0, 4, 6, 3, 0, 2, 0, 0]]),
 'edge_index': array([[0, 1, 1, 2, 1, 7, 2, 3, 3, 4, 3, 5, 3, 7, 5, 6, 7, 8],
        [1, 0, 2, 1, 7, 1, 3, 2, 4, 3, 5, 3, 7, 3, 6, 5, 8, 7]]),
 'edge_attr': array([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [2, 0, 0],
        [2, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'additional_x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 1, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
       

In [11]:
list_alchemy_lumo_data[0]

{'task': 'alchemy_lumo',
 'x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 1, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
        [5, 2, 4, 5, 0, 0, 2, 0, 1],
        [5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 0, 2, 5, 0, 0, 0, 0, 0],
        [5, 0, 2, 5, 1, 0, 0, 0, 0],
        [5, 2, 4, 5, 1, 0, 2, 0, 1],
        [6, 0, 4, 6, 3, 0, 2, 0, 0]]),
 'edge_index': array([[0, 1, 1, 2, 1, 7, 2, 3, 3, 4, 3, 5, 3, 7, 5, 6, 7, 8],
        [1, 0, 2, 1, 7, 1, 3, 2, 4, 3, 5, 3, 7, 3, 6, 5, 8, 7]]),
 'edge_attr': array([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [2, 0, 0],
        [2, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'additional_x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 1, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
       

In [12]:
list_alchemy_homo_lumo_gap_data[0]

{'task': 'alchemy_homo_lumo_gap',
 'x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 1, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1],
        [5, 2, 4, 5, 0, 0, 2, 0, 1],
        [5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 0, 2, 5, 0, 0, 0, 0, 0],
        [5, 0, 2, 5, 1, 0, 0, 0, 0],
        [5, 2, 4, 5, 1, 0, 2, 0, 1],
        [6, 0, 4, 6, 3, 0, 2, 0, 0]]),
 'edge_index': array([[0, 1, 1, 2, 1, 7, 2, 3, 3, 4, 3, 5, 3, 7, 5, 6, 7, 8],
        [1, 0, 2, 1, 7, 1, 3, 2, 4, 3, 5, 3, 7, 3, 6, 5, 8, 7]]),
 'edge_attr': array([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [2, 0, 0],
        [2, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'additional_x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 1, 4, 5, 1, 0, 2, 0, 1],
        [5, 0, 4, 5, 2, 0, 2, 0, 1]